# Audit des dimensions de qualité

**Objectif :** évaluer la qualité des données clients et produits selon
sept dimensions : complétude, validité, cohérence, exactitude, unicité,
fraîcheur et valeurs aberrantes.

Pour chaque dimension :
- les règles appliquées proviennent du référentiel `rules/rules_catalog.csv` ;
- le nombre et le taux de lignes concernées sont calculés ;
- les lignes à examiner sont listées ;
- l'impact métier et le niveau de certitude sont précisés.

| Résultat | Niveau de certitude | Signification |
|---|---|---|
| `FAIL` | Certaine | Une règle explicite (structure, format, domaine légal) est violée. |
| `REVIEW` | Probable | La règle repose sur une hypothèse métier à confirmer. |
| `INFO` | À vérifier | Signal statistique : une valeur extrême n'est pas une erreur confirmée. |

Aucune correction n'est appliquée à cette étape.

In [1]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import dq

datasets = dq.load_datasets()
clients_df = datasets["clients.csv"]
products_df = datasets["produits.csv"]

catalog = dq.load_rules_catalog()
context = dq.default_context(run_date=date.today())

rule_results, violations = dq.run_rules(datasets, catalog, context)

print(f"Run date: {context['run_date']}")
print(f"Rules executed: {len(rule_results)}")

Run date: 2026-11-24
Rules executed: 33


In [2]:
def show_rules(rule_ids):
    """Display the summary of the given rules, then the rows to review."""
    display(
        rule_results.loc[
            rule_results["rule_id"].isin(rule_ids),
            ["rule_id", "field", "rule", "rows_violating",
             "violation_rate_pct", "test_result"],
        ].reset_index(drop=True)
    )

    flagged = violations.loc[
        violations["rule_id"].isin(rule_ids),
        ["rule_id", "file", "row_number", "identifier",
         "field", "observed_value", "test_result"],
    ]
    if flagged.empty:
        print("No rows to review.")
    else:
        display(flagged.reset_index(drop=True))


def rule_ids(dimension):
    return catalog.loc[catalog["dimension"].eq(dimension), "rule_id"].tolist()

## 1. Complétude

**Règle de mesure :** une valeur est renseignée si elle n'est ni NULL,
ni une chaîne vide, ni une chaîne composée uniquement d'espaces.
Taux de complétude = valeurs renseignées / nombre de lignes.

**Remarque sur NULL et chaîne vide :** un fichier CSV ne distingue pas
NULL d'une chaîne vide (`;;`). Les fichiers étant lus en texte brut
(`keep_default_na=False`), une cellule vide apparaît comme `""` :
le nombre de NULL est donc nul par construction. La distinction NULL / vide
réapparaîtra lors de l'import SQL, selon la méthode d'import choisie.

In [3]:
# Completeness by column for the clients dataset
dq.missing_summary(clients_df)

,null_count,empty_string_count,whitespace_only_count,surrounding_spaces_count,populated_count,completeness_pct
id_client,0,0,0,0,10,100.0
prenom,0,0,0,0,10,100.0
nom,0,0,0,0,10,100.0
age,0,0,0,0,10,100.0
email,0,1,0,0,9,90.0
telephone,0,0,0,0,10,100.0
ville,0,0,0,0,10,100.0
code_postal,0,0,0,0,10,100.0
type_client,0,0,0,0,10,100.0
date_inscription,0,0,0,0,10,100.0


In [4]:
# Completeness by column for the products dataset
dq.missing_summary(products_df)

,null_count,empty_string_count,whitespace_only_count,surrounding_spaces_count,populated_count,completeness_pct
id_produit,0,0,0,0,100,100.0
nom_produit,0,0,0,0,100,100.0
categorie,0,1,0,0,99,99.0
conditionnement,0,0,0,0,100,100.0
prix_unitaire_eur,0,0,0,0,100,100.0
tva_pct,0,0,0,0,100,100.0
stock_unites,0,1,0,0,99,99.0
statut_stock,0,0,0,0,100,100.0
type_fournisseur,0,0,0,0,100,100.0
entreprise_exemple,0,0,0,0,100,100.0


In [5]:
# Overall completeness: populated cells / expected cells
for file_name, df in datasets.items():
    blank_cells = df.apply(dq.is_blank).to_numpy().sum()
    expected_cells = df.size
    rate = (expected_cells - blank_cells) / expected_cells * 100
    print(f"{file_name}: {expected_cells - blank_cells}/{expected_cells} "
          f"populated cells ({rate:.1f} %)")

clients.csv: 109/110 populated cells (99.1 %)
produits.csv: 998/1000 populated cells (99.8 %)


In [6]:
show_rules(rule_ids("Complétude"))

,rule_id,field,rule,rows_violating,violation_rate_pct,test_result
0,CLI-COM-01,prenom|nom|age|ville|code_postal|type_client|d...,Champs obligatoires renseignés,0,0.0,PASS
1,CLI-COM-02,email|telephone,Coordonnées de contact renseignées,1,10.0,REVIEW
2,PRD-COM-01,nom_produit|categorie|conditionnement|prix_uni...,Champs obligatoires renseignés,2,2.0,FAIL


,rule_id,file,row_number,identifier,field,observed_value,test_result
0,CLI-COM-02,clients.csv,3,CLI003,email,'',REVIEW
1,PRD-COM-01,produits.csv,5,PRD005,categorie,'',FAIL
2,PRD-COM-01,produits.csv,15,PRD015,stock_unites,'',FAIL


**Interprétation :**

| Ligne | Champ | Constat | Certitude | Impact métier |
|---|---|---|---|---|
| `CLI003` | `email` | E-mail absent | Probable (`REVIEW`) | Le client ne peut pas recevoir de communication par e-mail ; il reste joignable par téléphone. |
| `PRD005` | `categorie` | Catégorie absente | Certaine (`FAIL`) | Le produit disparaît des analyses par catégorie et peut être mal rangé dans le catalogue. |
| `PRD015` | `stock_unites` | Stock absent | Certaine (`FAIL`) | La disponibilité du produit est inconnue ; les statistiques de stock portent sur 99 produits. |

La complétude globale est élevée (plus de 99 %) : les problèmes de qualité
de ce jeu de données relèvent surtout des autres dimensions.
Aucune valeur manquante ne doit être remplacée sans source fiable
(fiche produit, inventaire ou contact du client).

## 2. Validité / conformité

**Règles appliquées :** formats (e-mail, téléphone, date, code postal)
et domaines de valeurs (âge, prix, stock, taux de TVA, modalités autorisées).
Ces contrôles vérifient qu'une valeur est *possible*, pas qu'elle est *vraie*.

In [7]:
catalog.loc[
    catalog["dimension"].eq("Validité"),
    ["rule_id", "field", "condition", "result_if_violated", "hypothesis"],
].reset_index(drop=True)

,rule_id,field,condition,result_if_violated,hypothesis
0,CLI-VAL-01,email,si renseigné : correspond à [^@\s]+@[^@\s]+\.[...,FAIL,"Contrôle de structure simple, sans vérificatio..."
1,CLI-VAL-02,telephone,"si renseigné, après suppression des espaces, p...",FAIL,Clients supposés joignables en France
2,CLI-VAL-03,date_inscription,si renseignée : format AAAA-MM-JJ et date cale...,FAIL,
3,CLI-VAL-04,age,si renseigné : nombre entier et 0 <= age <= 120,FAIL,120 ans : borne haute plausible (record humain...
4,CLI-VAL-05,age,si age numérique : age >= 18,REVIEW,"Hypothèse : un client inscrit (achat d'alcool,..."
5,CLI-VAL-06,code_postal,si renseigné : exactement 5 chiffres,FAIL,Adresses françaises uniquement
6,CLI-VAL-07,type_client,si renseigné : Particulier ou Professionnel,FAIL,"Liste des modalités observées, à confirmer par..."
7,PRD-VAL-01,prix_unitaire_eur,si renseigné : nombre et prix >= 0,REVIEW,Un prix nul est accepté si les produits gratui...
8,PRD-VAL-02,tva_pct,"si renseigné : taux dans {2.1, 5.5, 10, 20}",FAIL,Taux en vigueur en France métropolitaine (CGI)
9,PRD-VAL-03,stock_unites,si renseigné : nombre entier et stock >= 0,REVIEW,Les stocks négatifs ne sont pas autorisés


In [8]:
show_rules(rule_ids("Validité"))

,rule_id,field,rule,rows_violating,violation_rate_pct,test_result
0,CLI-VAL-01,email,Format d'e-mail valide,1,10.0,FAIL
1,CLI-VAL-02,telephone,Format de téléphone français valide,1,10.0,FAIL
2,CLI-VAL-03,date_inscription,Date d'inscription valide,0,0.0,PASS
3,CLI-VAL-04,age,Âge entier compris entre 0 et 120,2,20.0,FAIL
4,CLI-VAL-05,age,Client majeur,1,10.0,REVIEW
5,CLI-VAL-06,code_postal,Code postal à 5 chiffres,0,0.0,PASS
6,CLI-VAL-07,type_client,Type de client autorisé,0,0.0,PASS
7,PRD-VAL-01,prix_unitaire_eur,Prix numérique et non négatif,1,1.0,REVIEW
8,PRD-VAL-02,tva_pct,Taux de TVA légal,1,1.0,FAIL
9,PRD-VAL-03,stock_unites,Stock entier et non négatif,1,1.0,REVIEW


,rule_id,file,row_number,identifier,field,observed_value,test_result
0,CLI-VAL-01,clients.csv,4,CLI004,email,'hugo.petit@@example',FAIL
1,CLI-VAL-02,clients.csv,6,CLI006,telephone,'06 12',FAIL
2,CLI-VAL-04,clients.csv,5,CLI005,age,'150',FAIL
3,CLI-VAL-04,clients.csv,7,CLI007,age,'172',FAIL
4,CLI-VAL-05,clients.csv,9,CLI009,age,'0',REVIEW
5,PRD-VAL-01,produits.csv,10,PRD010,prix_unitaire_eur,'-5.2',REVIEW
6,PRD-VAL-02,produits.csv,30,PRD030,tva_pct,'99.0',FAIL
7,PRD-VAL-03,produits.csv,40,PRD040,stock_unites,'-12.0',REVIEW


**Interprétation :**

| Ligne | Champ | Valeur | Certitude | Impact métier |
|---|---|---|---|---|
| `CLI004` | `email` | `hugo.petit@@example` | Certaine | E-mail non délivrable : communications et factures perdues. |
| `CLI006` | `telephone` | `06 12` | Certaine | Numéro incomplet : client injoignable par téléphone. |
| `CLI005`, `CLI007` | `age` | 150 et 172 | Certaine | Âges impossibles qui faussent l'âge moyen et la segmentation. |
| `CLI009` | `age` | 0 | Probable | Probable valeur par défaut saisie à la place d'un âge inconnu. |
| `PRD010` | `prix_unitaire_eur` | -5,2 | Probable | Un prix négatif produirait un chiffre d'affaires négatif. |
| `PRD030` | `tva_pct` | 99 | Certaine | Taux inexistant en France : prix TTC et déclaration de TVA faux. |
| `PRD040` | `stock_unites` | -12 | Probable | Stock physiquement impossible, sauf si les ventes en rupture sont autorisées. |

Les dates d'inscription, les codes postaux, les types de clients
et les statuts de stock respectent leurs formats et domaines.

## 3. Cohérence

**Règles appliquées :** relations logiques entre champs.

- Ville / code postal : le code postal doit appartenir à la ville,
  d'après le référentiel `rules/ref_villes_cp.csv`.
- Date d'inscription : elle ne doit pas être postérieure à la date d'exécution.
- Stock / statut : un produit sans unité disponible ne peut pas être
  « En stock » ou « Stock faible ».
- Seuil de stock faible : « Stock faible » si et seulement si le stock
  est inférieur à 20 unités (seuil hypothétique à fixer par le métier).

In [9]:
# Reference table used for the city / postal code rule
dq.load_city_reference()

,ville,code_postal
0,Lille,59000
1,Lille,59160
2,Lille,59260
3,Lille,59777
4,Lille,59800
5,Roubaix,59100
6,Tourcoing,59200
7,Villeneuve-d'Ascq,59491
8,Villeneuve-d'Ascq,59493
9,Villeneuve-d'Ascq,59650


In [10]:
show_rules(rule_ids("Cohérence"))

,rule_id,field,rule,rows_violating,violation_rate_pct,test_result
0,CLI-COH-01,ville|code_postal,Code postal cohérent avec la ville,1,10.0,FAIL
1,CLI-COH-02,date_inscription,Date d'inscription non future,0,0.0,PASS
2,PRD-COH-01,stock_unites|statut_stock,Produit sans stock non déclaré disponible,2,2.0,FAIL
3,PRD-COH-02,stock_unites|statut_stock,Statut conforme au seuil de stock faible,28,28.0,REVIEW


,rule_id,file,row_number,identifier,field,observed_value,test_result
0,CLI-COH-01,clients.csv,8,CLI008,ville|code_postal,ville='Lens' | code_postal='75001',FAIL
1,PRD-COH-01,produits.csv,21,PRD021,stock_unites|statut_stock,stock_unites='0.0' | statut_stock='En stock',FAIL
2,PRD-COH-01,produits.csv,40,PRD040,stock_unites|statut_stock,stock_unites='-12.0' | statut_stock='En stock',FAIL
3,PRD-COH-02,produits.csv,4,PRD004,stock_unites|statut_stock,stock_unites='16.0' | statut_stock='En stock',REVIEW
4,PRD-COH-02,produits.csv,7,PRD007,stock_unites|statut_stock,stock_unites='64.0' | statut_stock='Stock faible',REVIEW
5,PRD-COH-02,produits.csv,8,PRD008,stock_unites|statut_stock,stock_unites='9.0' | statut_stock='En stock',REVIEW
6,PRD-COH-02,produits.csv,11,PRD011,stock_unites|statut_stock,stock_unites='31.0' | statut_stock='Stock faible',REVIEW
7,PRD-COH-02,produits.csv,16,PRD016,stock_unites|statut_stock,stock_unites='19.0' | statut_stock='En stock',REVIEW
8,PRD-COH-02,produits.csv,22,PRD022,stock_unites|statut_stock,stock_unites='49.0' | statut_stock='Stock faible',REVIEW
9,PRD-COH-02,produits.csv,25,PRD025,stock_unites|statut_stock,stock_unites='88.0' | statut_stock='Stock faible',REVIEW


Le grand nombre d'écarts à la règle du seuil conduit à vérifier
si `statut_stock` dépend réellement de `stock_unites`.

In [11]:
# Compare stock quantities by declared stock status
stock_by_status = (
    products_df
    .assign(stock=dq.to_number(products_df["stock_unites"]))
    .groupby("statut_stock")["stock"]
    .describe()
)
stock_by_status

,count,mean,std,min,25%,50%,75%,max
statut_stock,,,,,,,,
En stock,73.0,78.328767,47.576854,-12.0,34.00,77.0,116.0,158.0
Stock faible,26.0,82.500000,40.998780,19.0,48.25,75.0,113.5,157.0


Les catégories `Conserves` et `Conserves & terrines` sont proches :
leur contenu est comparé pour vérifier la cohérence de la nomenclature.

In [12]:
products_df.loc[
    products_df["categorie"].str.startswith("Conserves"),
    ["id_produit", "nom_produit", "categorie"],
]

,id_produit,nom_produit,categorie
32,PRD033,Terrine flamande,Conserves & terrines
33,PRD034,Terrine de campagne,Conserves & terrines
34,PRD035,Rillettes de porc,Conserves & terrines
35,PRD036,Pâté à la bière,Conserves & terrines
72,PRD073,Cornichons aigres-doux,Conserves
73,PRD074,Betteraves en conserve,Conserves
74,PRD075,Endives en conserve,Conserves


**Interprétation :**

- **Ville / code postal (certain) :** `CLI008` associe Lens au code
  `75001`, qui correspond à Paris (Lens : `62300`). L'une des deux valeurs
  est fausse, sans qu'il soit possible de savoir laquelle à partir du fichier.
  Impact : livraisons et analyses géographiques erronées.
  Arras (`62000`) est cohérent : le Pas-de-Calais fait partie
  des Hauts-de-France.
- **Stock nul ou négatif déclaré disponible (certain) :** `PRD021`
  (0 unité) et `PRD040` (-12 unités) sont « En stock ».
  Impact : vente de produits indisponibles.
- **Seuil de stock faible (probable) :** 28 des 97 produits ayant
  un stock positif contredisent le seuil de 20 unités. Surtout, le stock médian
  est presque identique pour « Stock faible » et « En stock » : le statut
  ne semble pas calculé à partir de la quantité. Il est peut-être saisi
  manuellement ou non mis à jour. Impact : réapprovisionnements mal priorisés.
- **Nomenclature (à vérifier) :** `Conserves` regroupe des légumes et
  `Conserves & terrines` des terrines et rillettes. Les deux libellés
  restent distincts, mais leur proximité peut entraîner des erreurs
  de classement. Ce point n'est pas compté comme une violation.
- Aucune date d'inscription n'est postérieure à la date d'exécution.

## 4. Exactitude

**Principe :** une valeur exacte correspond à la réalité. Une valeur peut
être valide (format et domaine respectés) mais inexacte. L'exactitude ne
peut donc pas être prouvée à partir du seul fichier : chaque anomalie
suspectée doit être confirmée avec une source de référence.

Les règles d'exactitude automatisables concernent les taux de TVA,
comparés à la réglementation fiscale.

In [13]:
show_rules(rule_ids("Exactitude"))

,rule_id,field,rule,rows_violating,violation_rate_pct,test_result
0,PRD-EXA-01,tva_pct,Boisson sans alcool au taux réduit,1,1.0,REVIEW
1,PRD-EXA-02,tva_pct,Carte cadeau sans TVA à l'émission,2,2.0,REVIEW
2,PRD-EXA-03,tva_pct,Confiserie au taux normal,6,6.0,REVIEW


,rule_id,file,row_number,identifier,field,observed_value,test_result
0,PRD-EXA-01,produits.csv,24,PRD024,tva_pct,'20.0',REVIEW
1,PRD-EXA-02,produits.csv,96,PRD096,tva_pct,'20.0',REVIEW
2,PRD-EXA-02,produits.csv,97,PRD097,tva_pct,'20.0',REVIEW
3,PRD-EXA-03,produits.csv,3,PRD003,tva_pct,'5.5',REVIEW
4,PRD-EXA-03,produits.csv,4,PRD004,tva_pct,'5.5',REVIEW
5,PRD-EXA-03,produits.csv,63,PRD063,tva_pct,'5.5',REVIEW
6,PRD-EXA-03,produits.csv,64,PRD064,tva_pct,'5.5',REVIEW
7,PRD-EXA-03,produits.csv,65,PRD065,tva_pct,'5.5',REVIEW
8,PRD-EXA-03,produits.csv,66,PRD066,tva_pct,'5.5',REVIEW


In [14]:
# Compare the suspicious gift box price with the other boxes and kits
products_df.loc[
    products_df["categorie"].eq("Coffrets & kits"),
    ["id_produit", "nom_produit", "conditionnement", "prix_unitaire_eur"],
]

,id_produit,nom_produit,conditionnement,prix_unitaire_eur
87,PRD088,Kit carbonnade flamande,kit 4 personnes,19.9
88,PRD089,Coffret découverte sucré,coffret 6 produits,24.9
89,PRD090,Coffret apéritif régional,coffret 5 produits,9999.9
90,PRD091,Panier gourmand petit format,panier 4 produits,34.9
91,PRD092,Panier gourmand grand format,panier 8 produits,59.9
99,PRD100,Coffret chicorée et biscuits,coffret 3 produits,18.9


**Anomalies d'exactitude suspectées et sources de confirmation :**

| Ligne | Valeur suspecte | Raisonnement | Certitude | Source de référence nécessaire |
|---|---|---|---|---|
| `PRD024` | TVA 20 % (bière sans alcool) | Les boissons non alcoolisées relèvent du taux réduit. | Probable | CGI / BOFiP, service comptable |
| `PRD096`, `PRD097` | TVA 20 % (cartes cadeaux) | Bon à usages multiples non soumis à TVA à l'émission. | Probable | CGI / BOFiP, service comptable |
| `PRD003`, `PRD004`, `PRD063` à `PRD066` | TVA 5,5 % (confiseries) | Les confiseries sont exclues du taux réduit. | Probable | CGI / BOFiP, service comptable |
| `PRD090` | Prix 9 999,90 € | Les autres coffrets coûtent entre 18,90 € et 59,90 €. | Probable | Fiche tarifaire, historique des ventes |
| `PRD010` | Prix -5,20 € | Les autres confitures coûtent entre 5,40 € et 5,60 € : possible erreur de signe. | Probable | Fiche tarifaire |
| `CLI008` | Lens / `75001` | Code postal de Paris. | Certaine (incohérence) | Base officielle des codes postaux, adresse du client |
| `CLI005`, `CLI007`, `CLI009` | Âges 150, 172 et 0 | Âges impossibles ou improbables. | Certaine / probable | Date de naissance déclarée par le client |
| `PRD040` | Stock -12 | Stock physique impossible. | Probable | Inventaire physique |
| `CLI001`, `PRD001` | Identifiants partagés | Impossible de savoir quelle entité porte le bon identifiant. | Certaine | Système source (ERP, CRM) |

Les valeurs proposées (par exemple 5,20 € au lieu de -5,20 €) ne sont que
des hypothèses : elles ne seront appliquées qu'après validation par la source.

## 5. Unicité

**Trois niveaux de contrôle :**

| Niveau | Règles | Question posée |
|---|---|---|
| Identifiant | `CLI-INT-02`, `PRD-INT-02` | Un identifiant désigne-t-il une seule ligne ? |
| Attributs | `CLI-UNI-01`, `CLI-UNI-02`, `PRD-UNI-01` | Une même entité apparaît-elle sous plusieurs identifiants ? |
| Ligne complète | `CLI-INT-03`, `PRD-INT-03` | Une ligne est-elle copiée à l'identique ? |

In [15]:
show_rules([
    "CLI-INT-02", "CLI-UNI-01", "CLI-UNI-02", "CLI-INT-03",
    "PRD-INT-02", "PRD-UNI-01", "PRD-INT-03",
])

,rule_id,field,rule,rows_violating,violation_rate_pct,test_result
0,CLI-INT-02,id_client,Identifiant client unique,2,20.0,FAIL
1,CLI-INT-03,all_columns,Aucune ligne entièrement dupliquée,0,0.0,PASS
2,CLI-UNI-01,email,E-mail non partagé,0,0.0,PASS
3,CLI-UNI-02,prenom|nom,Couple prénom + nom non répété,0,0.0,PASS
4,PRD-INT-02,id_produit,Identifiant produit unique,2,2.0,FAIL
5,PRD-INT-03,all_columns,Aucune ligne entièrement dupliquée,0,0.0,PASS
6,PRD-UNI-01,nom_produit|conditionnement,Couple nom + conditionnement non répété,0,0.0,PASS


,rule_id,file,row_number,identifier,field,observed_value,test_result
0,CLI-INT-02,clients.csv,1,CLI001,id_client,'CLI001',FAIL
1,CLI-INT-02,clients.csv,2,CLI001,id_client,'CLI001',FAIL
2,PRD-INT-02,produits.csv,1,PRD001,id_produit,'PRD001',FAIL
3,PRD-INT-02,produits.csv,2,PRD001,id_produit,'PRD001',FAIL


In [16]:
# Compare the rows sharing the same identifier
display(clients_df.loc[clients_df["id_client"].eq("CLI001")])
display(products_df.loc[products_df["id_produit"].eq("PRD001")])

,id_client,prenom,nom,age,email,telephone,ville,code_postal,type_client,date_inscription,entreprise_exemple
0,CLI001,Camille,Leroy,29,camille.leroy@example.com,06 12 34 50 01,Lille,59000,Particulier,2025-02-14,Les Saveurs du Nord (entreprise fictive)
1,CLI001,Thomas,Dubois,41,thomas.dubois@example.com,06 12 34 50 02,Roubaix,59100,Professionnel,2024-11-06,Les Saveurs du Nord (entreprise fictive)


,id_produit,nom_produit,categorie,conditionnement,prix_unitaire_eur,tva_pct,stock_unites,statut_stock,type_fournisseur,entreprise_exemple
0,PRD001,Gaufres artisanales,Biscuits & douceurs,paquet 6 pièces,4.2,5.5,36.0,En stock,Fournisseur alimentaire,Les Saveurs du Nord (entreprise fictive)
1,PRD001,Spéculoos pur beurre,Biscuits & douceurs,sachet 250 g,3.6,5.5,70.0,En stock,Artisan partenaire,Les Saveurs du Nord (entreprise fictive)


**Interprétation :**

- Les lignes `CLI001` concernent deux personnes différentes (noms, e-mails,
  villes et types différents). Les lignes `PRD001` concernent deux produits
  différents (Gaufres artisanales et Spéculoos pur beurre).
- Il ne s'agit donc **pas de doublons d'entités** mais de **deux entités
  distinctes partageant un identifiant** : supprimer l'une des lignes
  ferait perdre un client ou un produit réel. La correction consiste
  à attribuer un nouvel identifiant à l'une des deux lignes.
- Aucun doublon d'entité n'est détecté sur les attributs (même e-mail,
  même prénom et nom, ou même produit dans le même conditionnement),
  ni aucune ligne entièrement dupliquée.

## 6. Fraîcheur / actualité

**Constat :** aucun des deux fichiers ne contient de date de mise à jour
des enregistrements.

- Clients : `date_inscription` indique la date de création du client,
  pas la date de dernière mise à jour de ses informations.
- Produits : aucune colonne de date. Le prix, le stock et le statut
  ne peuvent pas être datés.
- La date de modification des fichiers CSV n'est pas fiable : elle change
  à chaque copie.

In [17]:
# Registration dates describe client seniority, not data freshness
registration = pd.to_datetime(clients_df["date_inscription"], format="%Y-%m-%d")
run_date = pd.Timestamp(context["run_date"])

print(f"First registration: {registration.min().date()}")
print(f"Last registration: {registration.max().date()}")
print(f"Days since the last registration: {(run_date - registration.max()).days}")

First registration: 2024-09-18
Last registration: 2025-06-03
Days since the last registration: 539


**Conclusion :** la fraîcheur n'est **pas mesurable** avec les données
fournies. Aucun indicateur de fraîcheur n'est calculé, pour ne pas
présenter un résultat inventé.

L'écart entre le stock et son statut pourrait être un symptôme de données
non mises à jour, mais cette hypothèse ne peut pas être vérifiée.

**Recommandation :** ajouter des colonnes `date_mise_a_jour`
(clients et produits) et `date_inventaire` (stock).

## 7. Valeurs aberrantes

**Méthode :** bornes de Tukey, sans hypothèse sur la forme de la distribution.
Une valeur est extrême si elle sort de l'intervalle
[Q1 - 1,5 x IQR ; Q3 + 1,5 x IQR], où IQR = Q3 - Q1.

Une valeur extrême est un **signal statistique à investiguer** (`INFO`),
pas une erreur confirmée. Elle devient une erreur certaine seulement
si elle viole aussi une règle explicite.

In [18]:
# Tukey bounds for each numerical column
bounds = []
for file_name, column in [
    ("clients.csv", "age"),
    ("produits.csv", "prix_unitaire_eur"),
    ("produits.csv", "stock_unites"),
]:
    values = dq.to_number(datasets[file_name][column]).dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    low, high = dq.tukey_bounds(values)
    bounds.append({
        "file": file_name,
        "field": column,
        "values": len(values),
        "q1": q1,
        "q3": q3,
        "iqr": q3 - q1,
        "lower_bound": round(low, 2),
        "upper_bound": round(high, 2),
        "outliers": int((values.lt(low) | values.gt(high)).sum()),
    })

pd.DataFrame(bounds)

,file,field,values,q1,q3,iqr,lower_bound,upper_bound,outliers
0,clients.csv,age,10,35.0,48.250,13.250,15.12,68.12,3
1,produits.csv,prix_unitaire_eur,100,3.8,6.425,2.625,-0.14,10.36,12
2,produits.csv,stock_unites,99,36.0,116.000,80.000,-84.00,236.00,0


In [19]:
show_rules(rule_ids("Valeurs aberrantes"))

,rule_id,field,rule,rows_violating,violation_rate_pct,test_result
0,CLI-OUT-01,age,Âge dans les bornes de Tukey,3,30.0,INFO
1,PRD-OUT-01,prix_unitaire_eur,Prix dans les bornes de Tukey,12,12.0,INFO
2,PRD-OUT-02,stock_unites,Stock dans les bornes de Tukey,0,0.0,PASS


,rule_id,file,row_number,identifier,field,observed_value,test_result
0,CLI-OUT-01,clients.csv,5,CLI005,age,'150',INFO
1,CLI-OUT-01,clients.csv,7,CLI007,age,'172',INFO
2,CLI-OUT-01,clients.csv,9,CLI009,age,'0',INFO
3,PRD-OUT-01,produits.csv,10,PRD010,prix_unitaire_eur,'-5.2',INFO
4,PRD-OUT-01,produits.csv,19,PRD019,prix_unitaire_eur,'12.9',INFO
5,PRD-OUT-01,produits.csv,37,PRD037,prix_unitaire_eur,'12.9',INFO
6,PRD-OUT-01,produits.csv,38,PRD038,prix_unitaire_eur,'10.9',INFO
7,PRD-OUT-01,produits.csv,88,PRD088,prix_unitaire_eur,'19.9',INFO
8,PRD-OUT-01,produits.csv,89,PRD089,prix_unitaire_eur,'24.9',INFO
9,PRD-OUT-01,produits.csv,90,PRD090,prix_unitaire_eur,'9999.9',INFO


In [20]:
# Cross the outliers with the explicit rules to separate errors from extremes
outliers = violations.loc[violations["dimension"].eq("Valeurs aberrantes")]
explicit = violations.loc[violations["test_result"].isin(["FAIL", "REVIEW"])]

explicit_by_row = (
    explicit.groupby(["file", "row_number"])["rule_id"]
    .agg(", ".join)
    .rename("explicit_rules_violated")
)

outliers[["rule_id", "file", "row_number", "identifier", "observed_value"]].join(
    explicit_by_row, on=["file", "row_number"]
).fillna({"explicit_rules_violated": "-"}).reset_index(drop=True)

,rule_id,file,row_number,identifier,observed_value,explicit_rules_violated
0,CLI-OUT-01,clients.csv,5,CLI005,'150',CLI-VAL-04
1,CLI-OUT-01,clients.csv,7,CLI007,'172',CLI-VAL-04
2,CLI-OUT-01,clients.csv,9,CLI009,'0',CLI-VAL-05
3,PRD-OUT-01,produits.csv,10,PRD010,'-5.2',PRD-VAL-01
4,PRD-OUT-01,produits.csv,19,PRD019,'12.9',-
5,PRD-OUT-01,produits.csv,37,PRD037,'12.9',-
6,PRD-OUT-01,produits.csv,38,PRD038,'10.9',-
7,PRD-OUT-01,produits.csv,88,PRD088,'19.9',PRD-COH-02
8,PRD-OUT-01,produits.csv,89,PRD089,'24.9',PRD-COH-02
9,PRD-OUT-01,produits.csv,90,PRD090,'9999.9',-


**Interprétation :**

- **Âges :** les 3 valeurs extrêmes (0, 150 et 172) violent aussi les règles
  de validité : ce sont des anomalies certaines ou probables. Avec seulement
  10 clients, les bornes de Tukey restent fragiles : les bornes métier
  (0 à 120 ans, majorité) sont plus fiables.
- **Prix :** 12 prix sont extrêmes, mais la plupart sont **normaux** :
  coffrets, paniers, kits, cartes cadeaux, café en grains (500 g)
  et plats préparés en bocal sont naturellement plus chers
  que les produits d'épicerie. Seuls -5,20 € (`PRD010`, règle de validité)
  et 9 999,90 € (`PRD090`, écart avec les autres coffrets) sont à traiter.
  Une analyse des prix par catégorie serait plus pertinente qu'une borne globale.
- **Stocks :** aucune valeur extrême. Le stock de -12 unités reste
  dans les bornes statistiques, mais viole la règle de non-négativité :
  **une méthode statistique ne remplace pas les règles métier.**

## 8. Synthèse et export

In [21]:
# Summary by dimension
dimension_order = [
    "Intégrité", "Complétude", "Validité", "Cohérence",
    "Exactitude", "Unicité", "Valeurs aberrantes",
]

summary = (
    rule_results
    .assign(rule_violated=rule_results["rows_violating"].gt(0))
    .groupby("dimension")
    .agg(
        rules_executed=("rule_id", "count"),
        rules_violated=("rule_violated", "sum"),
        rows_flagged=("rows_violating", "sum"),
    )
    .reindex(dimension_order)
)
summary

,rules_executed,rules_violated,rows_flagged
dimension,,,
Intégrité,6,2,4
Complétude,3,2,3
Validité,11,7,8
Cohérence,4,3,31
Exactitude,3,3,9
Unicité,3,0,0
Valeurs aberrantes,3,2,15


In [22]:
# Number of flagged rows by result level and file
pd.crosstab(
    violations["file"],
    violations["test_result"],
    margins=True,
    margins_name="Total",
)

test_result,FAIL,INFO,REVIEW,Total
file,,,,
clients.csv,7,3,2,12
produits.csv,7,12,39,58
Total,14,15,41,70


In [23]:
# Export the rule results and the anomaly tracking table
dq.OUTPUT_DIR.mkdir(exist_ok=True)

rule_results.to_csv(
    dq.OUTPUT_DIR / "rule_results.csv", sep=";", index=False, encoding="utf-8-sig"
)
violations.to_csv(
    dq.OUTPUT_DIR / "anomalies.csv", sep=";", index=False, encoding="utf-8-sig"
)

print(f"rule_results.csv: {len(rule_results)} rules")
print(f"anomalies.csv: {len(violations)} rows to review")

rule_results.csv: 33 rules
anomalies.csv: 70 rows to review


**Remarque :** `rows_flagged` additionne les lignes signalées par chaque
règle. Une même ligne peut être signalée par plusieurs règles : par exemple,
`PRD040` (stock négatif et stock déclaré disponible) ou `PRD010`
(prix négatif et prix extrême).

**Principaux constats :**

| Dimension | Constat principal | Certitude |
|---|---|---|
| Intégrité / unicité | `CLI001` et `PRD001` désignent chacun deux entités distinctes. | Certaine |
| Complétude | Plus de 99 % des valeurs renseignées ; 1 e-mail, 1 catégorie et 1 stock absents. | Certaine / probable |
| Validité | E-mail et téléphone invalides, âges impossibles, TVA à 99 %, prix et stock négatifs. | Certaine / probable |
| Cohérence | Code postal parisien pour Lens ; statut de stock indépendant de la quantité. | Certaine / probable |
| Exactitude | 9 taux de TVA probablement erronés ; prix de 9 999,90 € suspect. | Probable |
| Fraîcheur | Non mesurable : aucune date de mise à jour. | Limite documentée |
| Valeurs aberrantes | Prix extrêmes surtout légitimes (coffrets, cartes cadeaux). | À vérifier |

Les fichiers exportés dans `outputs/` alimenteront les parties SQL,
Python et tableau de bord.